# 机器学习概述（下）Runner 类与加州房价案例

本节做两件事：

1. 抽象一个 **Runner** 类，把训练循环、评估、保存/加载封装起来——后续章节会持续复用并扩展它。
2. 跑通**加州房价回归**的端到端案例：sklearn 拉数据 → 训练/测试划分 → 特征归一化 → 训练 → 评估 → 预测。

## 1. Runner：一个最小训练骨架

PyTorch 的训练循环模板很固定：

```python
for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader:
        loss = loss_fn(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    eval_metric = evaluate(model, dev_loader)
```

我们把它包成 `Runner`，约定接口：`fit / evaluate / predict / save / load`。

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


class Runner:
    """训练 / 评估 / 保存 / 加载 的最小封装。

    后续章节会扩展：early stopping、metrics 记录、学习率调度等。
    """

    def __init__(self, model, optimizer, loss_fn, metric_fn=None):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.metric_fn = metric_fn      # 缺省时用 loss 当指标
        self.history = {'train_loss': [], 'dev_metric': []}

    def fit(self, train_loader, dev_loader=None, num_epochs=100, log_every=10):
        for epoch in range(num_epochs):
            self.model.train()
            running = 0.0
            for x, y in train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(x), y)
                loss.backward()
                self.optimizer.step()
                running += loss.item() * x.size(0)
            train_loss = running / len(train_loader.dataset)
            self.history['train_loss'].append(train_loss)

            if dev_loader is not None:
                dev_m = self.evaluate(dev_loader)
                self.history['dev_metric'].append(dev_m)
                if (epoch + 1) % log_every == 0:
                    print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}  dev_metric={dev_m:.4f}')
            elif (epoch + 1) % log_every == 0:
                print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}')

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total, n = 0.0, 0
        fn = self.metric_fn if self.metric_fn is not None else self.loss_fn
        for x, y in loader:
            total += fn(self.model(x), y).item() * x.size(0)
            n += x.size(0)
        return total / n

    @torch.no_grad()
    def predict(self, x):
        self.model.eval()
        return self.model(x)

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path, weights_only=True))

## 2. 加州房价回归

数据集来自 sklearn 自带 `fetch_california_housing`：20640 个样本、8 个特征（中位收入、房龄、平均房间数等），目标是该地区房价中位数（单位 $100k）。

和波士顿房价数据集相比，加州数据集：样本量更大、没有缺失值、没有伦理争议的列，是当前推荐的回归教学数据集。

### 2.1 加载数据

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

ds = fetch_california_housing(as_frame=True)
df = ds.frame                # DataFrame: 8 特征 + 1 target (MedHouseVal)
print('shape:', df.shape)
print('missing per column:', df.isna().sum().sum(), '(no NaN)')
df.head()

特征含义：

| 列 | 含义 |
|---|---|
| `MedInc` | 街区中位家庭收入（单位 $10k） |
| `HouseAge` | 街区房屋中位年龄 |
| `AveRooms` | 平均房间数 |
| `AveBedrms` | 平均卧室数 |
| `Population` | 街区人口 |
| `AveOccup` | 户均人口 |
| `Latitude` / `Longitude` | 街区地理坐标 |
| `MedHouseVal` | **目标**：房价中位数（$100k） |

### 2.2 划分训练 / 测试集 + 特征归一化

**关键约定**：归一化的 `mean/std` 只能在训练集上拟合，再用同一组统计量去变换测试集——否则会发生测试集信息泄露。

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

X = torch.tensor(df.drop(columns=['MedHouseVal']).values, dtype=torch.float32)
y = torch.tensor(df['MedHouseVal'].values, dtype=torch.float32).unsqueeze(1)

N = len(X)
perm = torch.randperm(N)
split = int(N * 0.8)
tr, te = perm[:split], perm[split:]
X_train, y_train = X[tr], y[tr]
X_test,  y_test  = X[te], y[te]

# mean/std 只在训练集上拟合
mean, std = X_train.mean(dim=0), X_train.std(dim=0)
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=128)
print(f'train: {len(X_train)}  test: {len(X_test)}  feature dim: {X_train.shape[1]}')

### 2.3 模型 + Runner + 训练

线性回归 = `nn.Linear(8, 1)`。用 MSE 作为损失和评估指标。

In [ ]:
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

model = nn.Linear(X_train.shape[1], 1)
optimizer = optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

runner = Runner(model, optimizer, loss_fn)
runner.fit(train_loader, dev_loader=test_loader, num_epochs=50, log_every=10)

### 2.4 保存与加载

用 `Path` 拼绝对路径，避免 notebook 在不同 CWD 下表现不一致。

In [ ]:
MODEL_DIR = Path.cwd() / 'models'
save_path = MODEL_DIR / 'california_linear.pt'
runner.save(save_path)

# 新建一个模型，从磁盘加载参数
model2 = nn.Linear(X_train.shape[1], 1)
runner2 = Runner(model2, optim.Adam(model2.parameters()), loss_fn)
runner2.load(save_path)
print('test MSE (reloaded):', runner2.evaluate(test_loader))

### 2.5 检查学到的权重

在归一化后的特征上，权重正负就能定性解释——负权重表示"该特征增大 → 房价下降"。

In [ ]:
feat_names = list(df.drop(columns=['MedHouseVal']).columns)
weights = model.weight.detach().squeeze().tolist()
bias = model.bias.item()

for name, w in sorted(zip(feat_names, weights), key=lambda kv: kv[1]):
    print(f'{name:12s}  {w:+.4f}')
print(f'\nbias         {bias:+.4f}')

典型观察：`MedInc`（中位收入）权重最正——收入越高的街区房价越高；`Latitude` 权重为负反映出加州北部（纬度高）房价低于南部沿海。

### 2.6 单点预测

In [ ]:
x0, y0 = X_test[:1], y_test[:1]
with torch.no_grad():
    pred = runner.predict(x0)
print(f'true price (×$100k): {y0.item():.2f}  →  pred: {pred.item():.2f}')

### 2.7 训练曲线

In [ ]:
import matplotlib.pyplot as plt

plt.plot(runner.history['train_loss'], label='train loss')
plt.plot(runner.history['dev_metric'], label='test MSE')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 小结

- **Runner** 把训练循环、评估、模型 IO 封装成一个对象——后续章节会扩展（验证集、早停、metric 记录、学习率调度）。
- **特征归一化** 的统计量必须只在训练集上拟合，再去变换测试集。
- **`state_dict` 保存/加载** 是 PyTorch 的官方推荐做法（比直接 pickle 整个模型更稳）。
- **数据集选择**：加州房价是当前 sklearn 推荐的回归示范数据集；波士顿房价已在 sklearn 1.2 中移除。